<a href="https://colab.research.google.com/github/Shivambharti28/Movie-Recommendation-System/blob/main/Movies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from sklearn.preprocessing import StandardScaler

In [ ]:
df = pd.read_csv('../data/AI_movies_dataset.csv', low_memory=False)
df.head()

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0


In [3]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  str    
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [5]:
df.shape

(45466, 24)

In [6]:
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [7]:
df = df.drop_duplicates().reset_index(drop=True)

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:

# 2. Function to parse JSON-like columns
def parse_json_column(column_data, key_name='name'):
    try:
        # Convert string representation of list to actual list
        items = ast.literal_eval(column_data)
        if isinstance(items, list):
            return ", ".join([i[key_name] for i in items if key_name in i])
        return "Unknown"
    except (ValueError, SyntaxError, TypeError):
        return "Unknown"
    
# Clean JSON-like columns
json_cols = ['genres', 'production_companies', 'production_countries', 'spoken_languages']
for col in json_cols:
    print(f"Processing JSON structure in: {col}")
    df[col] = df[col].apply(lambda x: parse_json_column(x) if pd.notnull(x) else "Unknown")

Processing JSON structure in: genres
Processing JSON structure in: production_companies
Processing JSON structure in: production_countries
Processing JSON structure in: spoken_languages


In [10]:
df

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"Animation, Comedy, Family",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,English,Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"Adventure, Fantasy, Family",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"English, Français",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"Romance, Comedy",NaN,15602,tt0113228,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,...,1995-12-22,0.0,101.0,English,Released,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,False,6.5,92.0
3,False,NaN,16000000,"Comedy, Drama, Romance",NaN,31357,tt0114885,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",...,1995-12-22,81452156.0,127.0,English,Released,Friends are the people who let you be yourself...,Waiting to Exhale,False,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,Comedy,NaN,11862,tt0113041,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,...,1995-02-10,76578911.0,106.0,English,Released,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,False,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45444,False,NaN,0,"Drama, Family",http://www.imdb.com/title/tt6209470/,439050,tt6209470,fa,رگ خواب,Rising and falling between a man and woman.,...,NaN,0.0,90.0,فارسی,Released,Rising and falling between a man and woman,Subdue,False,4.0,1.0
45445,False,NaN,0,Drama,NaN,111109,tt2028550,tl,Siglo ng Pagluluwal,An artist struggles to finish his work while a...,...,2011-11-17,0.0,360.0,,Released,NaN,Century of Birthing,False,9.0,3.0
45446,False,NaN,0,"Action, Drama, Thriller",NaN,67758,tt0303758,en,Betrayal,"When one of her hits goes wrong, a professiona...",...,2003-08-01,0.0,90.0,English,Released,A deadly game of wits.,Betrayal,False,3.8,6.0
45447,False,NaN,0,,NaN,227506,tt0008536,en,Satana likuyushchiy,"In a small town live two brothers, one a minis...",...,1917-10-21,0.0,87.0,,Released,NaN,Satan Triumphant,False,0.0,0.0


In [11]:
df.isnull().sum()

adult                        0
belongs_to_collection    40956
budget                       0
genres                       0
homepage                 37669
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         0
production_countries         0
release_date                87
revenue                      6
runtime                    263
spoken_languages             0
status                      87
tagline                  25042
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [12]:
# 3. Data Cleaning: Robust Type Conversion
# Convert problematic columns to numeric, forcing errors to NaN
df['budget'] = pd.to_numeric(df['budget'], errors='coerce')
df['popularity'] = pd.to_numeric(df['popularity'], errors='coerce')



# 4. Handling Missing Values

df = df.dropna(subset=['title']) # --> droping rows of title with null values
df['tagline'] = df['tagline'].fillna(' ') # --> placing blank space in place of null values in tagline
df.dropna(subset = ['release_date'],inplace=True)


In [13]:
df['overview'] = df['overview'].fillna(' ') # --> placing blank space in place of null values in overview

In [14]:
# Fill numerical missing values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill categorical missing values with 'Unknown'
# object_cols = df.select_dtypes(include=['object']).columns
# for col in object_cols:
#     df[col] = df[col].fillna("Unknown")

# Remove duplicates
df.drop_duplicates(inplace=True)

df.drop(columns = ["homepage","imdb_id","original_language","original_title","poster_path","video","status"],inplace = True)

In [15]:
# 5. Statistical Overview
print("\nCleaned Dataset Shape:", df.shape)
display(df)
display(df.info())



Cleaned Dataset Shape: (45359, 17)


,adult,belongs_to_collection,budget,genres,id,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000.0,"Animation, Comedy, Family",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,Pixar Animation Studios,United States of America,1995-10-30,373554033.0,81.0,English,,Toy Story,7.7,5415.0
1,False,NaN,65000000.0,"Adventure, Fantasy, Family",8844,When siblings Judy and Peter discover an encha...,17.015539,"TriStar Pictures, Teitler Film, Interscope Com...",United States of America,1995-12-15,262797249.0,104.0,"English, Français",Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0
2,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0.0,"Romance, Comedy",15602,A family wedding reignites the ancient feud be...,11.712900,"Warner Bros., Lancaster Gate",United States of America,1995-12-22,0.0,101.0,English,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0
3,False,NaN,16000000.0,"Comedy, Drama, Romance",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,Twentieth Century Fox Film Corporation,United States of America,1995-12-22,81452156.0,127.0,English,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0
4,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0.0,Comedy,11862,Just when George Banks has recovered from his ...,8.387519,"Sandollar Productions, Touchstone Pictures",United States of America,1995-02-10,76578911.0,106.0,English,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45443,False,NaN,0.0,"Drama, Action, Romance",30840,"Yet another version of the classic epic, with ...",5.683753,"Westdeutscher Rundfunk (WDR), Working Title Fi...","Canada, Germany, United Kingdom, United States...",1991-05-13,0.0,104.0,English,,Robin Hood,5.7,26.0
45445,False,NaN,0.0,Drama,111109,An artist struggles to finish his work while a...,0.178241,Sine Olivia,Philippines,2011-11-17,0.0,360.0,,,Century of Birthing,9.0,3.0
45446,False,NaN,0.0,"Action, Drama, Thriller",67758,"When one of her hits goes wrong, a professiona...",0.903007,American World Pictures,United States of America,2003-08-01,0.0,90.0,English,A deadly game of wits.,Betrayal,3.8,6.0
45447,False,NaN,0.0,,227506,"In a small town live two brothers, one a minis...",0.003503,Yermoliev,Russia,1917-10-21,0.0,87.0,,,Satan Triumphant,0.0,0.0


<class 'pandas.DataFrame'>
Index: 45359 entries, 0 to 45448
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45359 non-null  str    
 1   belongs_to_collection  4487 non-null   str    
 2   budget                 45359 non-null  float64
 3   genres                 45359 non-null  str    
 4   id                     45359 non-null  str    
 5   overview               45359 non-null  str    
 6   popularity             45359 non-null  float64
 7   production_companies   45359 non-null  str    
 8   production_countries   45359 non-null  str    
 9   release_date           45359 non-null  str    
 10  revenue                45359 non-null  float64
 11  runtime                45359 non-null  float64
 12  spoken_languages       45359 non-null  str    
 13  tagline                45359 non-null  str    
 14  title                  45359 non-null  str    
 15  vote_average      

None

In [16]:
df.isnull().sum()

adult                        0
belongs_to_collection    40872
budget                       0
genres                       0
id                           0
overview                     0
popularity                   0
production_companies         0
production_countries         0
release_date                 0
revenue                      0
runtime                      0
spoken_languages             0
tagline                      0
title                        0
vote_average                 0
vote_count                   0
dtype: int64

In [17]:
import ast

# Function to safely extract the collection name
def fetch_collection(text):
    if pd.isna(text):
        return " "
    try:
        # The data is stored as a stringified dictionary
        return ast.literal_eval(text)['name']
    except:
        return " "

# 1. Create a new collection column
df['collection'] = df['belongs_to_collection'].apply(fetch_collection)

# 2. Update your tags creation to INCLUDE the collection!
df['tags'] = df['overview'] + " " + df['genres'] + " " + df['tagline'] + " " + df['collection'] + df['production_companies'] + df["production_countries"]

df.drop(columns = ["belongs_to_collection"],inplace=True)

In [18]:
df.head()

,adult,budget,genres,id,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count,collection,tags
0,False,30000000.0,"Animation, Comedy, Family",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,Pixar Animation Studios,United States of America,1995-10-30,373554033.0,81.0,English,,Toy Story,7.7,5415.0,Toy Story Collection,"Led by Woody, Andy's toys live happily in his ..."
1,False,65000000.0,"Adventure, Fantasy, Family",8844,When siblings Judy and Peter discover an encha...,17.015539,"TriStar Pictures, Teitler Film, Interscope Com...",United States of America,1995-12-15,262797249.0,104.0,"English, Français",Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0,,When siblings Judy and Peter discover an encha...
2,False,0.0,"Romance, Comedy",15602,A family wedding reignites the ancient feud be...,11.712900,"Warner Bros., Lancaster Gate",United States of America,1995-12-22,0.0,101.0,English,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0,Grumpy Old Men Collection,A family wedding reignites the ancient feud be...
3,False,16000000.0,"Comedy, Drama, Romance",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,Twentieth Century Fox Film Corporation,United States of America,1995-12-22,81452156.0,127.0,English,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0,,"Cheated on, mistreated and stepped on, the wom..."
4,False,0.0,Comedy,11862,Just when George Banks has recovered from his ...,8.387519,"Sandollar Productions, Touchstone Pictures",United States of America,1995-02-10,76578911.0,106.0,English,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0,Father of the Bride Collection,Just when George Banks has recovered from his ...


# **NATURAL LANGUAGE PROCESSING**





In [19]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

In [20]:
# nltk.download('stopwords')
# nltk.download('wordnet')

In [21]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [22]:
def preprocess_text(text):
  text = str(text).lower() # --> lowercase
  text = re.sub(r'[^a-zA-Z\s]', '',text) # --> removed the punctuations

  words = text.split()

  # remove stopwords
  words = [word for word in words if word not in stop_words]

  #lemmatize
  words = [lemmatizer.lemmatize(word) for word in words]

  return " ".join(words) # --> list to string conversion

In [23]:
df['tags'] = df['tags'].apply(preprocess_text)

In [24]:
df.head()

,adult,budget,genres,id,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count,collection,tags
0,False,30000000.0,"Animation, Comedy, Family",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,Pixar Animation Studios,United States of America,1995-10-30,373554033.0,81.0,English,,Toy Story,7.7,5415.0,Toy Story Collection,led woody andys toy live happily room andys bi...
1,False,65000000.0,"Adventure, Fantasy, Family",8844,When siblings Judy and Peter discover an encha...,17.015539,"TriStar Pictures, Teitler Film, Interscope Com...",United States of America,1995-12-15,262797249.0,104.0,"English, Français",Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0,,sibling judy peter discover enchanted board ga...
2,False,0.0,"Romance, Comedy",15602,A family wedding reignites the ancient feud be...,11.712900,"Warner Bros., Lancaster Gate",United States of America,1995-12-22,0.0,101.0,English,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0,Grumpy Old Men Collection,family wedding reignites ancient feud nextdoor...
3,False,16000000.0,"Comedy, Drama, Romance",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,Twentieth Century Fox Film Corporation,United States of America,1995-12-22,81452156.0,127.0,English,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0,,cheated mistreated stepped woman holding breat...
4,False,0.0,Comedy,11862,Just when George Banks has recovered from his ...,8.387519,"Sandollar Productions, Touchstone Pictures",United States of America,1995-02-10,76578911.0,106.0,English,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0,Father of the Bride Collection,george bank recovered daughter wedding receive...


In [25]:
df = df.reset_index(drop = True)

In [26]:
df

,adult,budget,genres,id,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,tagline,title,vote_average,vote_count,collection,tags
0,False,30000000.0,"Animation, Comedy, Family",862,"Led by Woody, Andy's toys live happily in his ...",21.946943,Pixar Animation Studios,United States of America,1995-10-30,373554033.0,81.0,English,,Toy Story,7.7,5415.0,Toy Story Collection,led woody andys toy live happily room andys bi...
1,False,65000000.0,"Adventure, Fantasy, Family",8844,When siblings Judy and Peter discover an encha...,17.015539,"TriStar Pictures, Teitler Film, Interscope Com...",United States of America,1995-12-15,262797249.0,104.0,"English, Français",Roll the dice and unleash the excitement!,Jumanji,6.9,2413.0,,sibling judy peter discover enchanted board ga...
2,False,0.0,"Romance, Comedy",15602,A family wedding reignites the ancient feud be...,11.712900,"Warner Bros., Lancaster Gate",United States of America,1995-12-22,0.0,101.0,English,Still Yelling. Still Fighting. Still Ready for...,Grumpier Old Men,6.5,92.0,Grumpy Old Men Collection,family wedding reignites ancient feud nextdoor...
3,False,16000000.0,"Comedy, Drama, Romance",31357,"Cheated on, mistreated and stepped on, the wom...",3.859495,Twentieth Century Fox Film Corporation,United States of America,1995-12-22,81452156.0,127.0,English,Friends are the people who let you be yourself...,Waiting to Exhale,6.1,34.0,,cheated mistreated stepped woman holding breat...
4,False,0.0,Comedy,11862,Just when George Banks has recovered from his ...,8.387519,"Sandollar Productions, Touchstone Pictures",United States of America,1995-02-10,76578911.0,106.0,English,Just When His World Is Back To Normal... He's ...,Father of the Bride Part II,5.7,173.0,Father of the Bride Collection,george bank recovered daughter wedding receive...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45354,False,0.0,"Drama, Action, Romance",30840,"Yet another version of the classic epic, with ...",5.683753,"Westdeutscher Rundfunk (WDR), Working Title Fi...","Canada, Germany, United Kingdom, United States...",1991-05-13,0.0,104.0,English,,Robin Hood,5.7,26.0,,yet another version classic epic enough variat...
45355,False,0.0,Drama,111109,An artist struggles to finish his work while a...,0.178241,Sine Olivia,Philippines,2011-11-17,0.0,360.0,,,Century of Birthing,9.0,3.0,,artist struggle finish work storyline cult pla...
45356,False,0.0,"Action, Drama, Thriller",67758,"When one of her hits goes wrong, a professiona...",0.903007,American World Pictures,United States of America,2003-08-01,0.0,90.0,English,A deadly game of wits.,Betrayal,3.8,6.0,,one hit go wrong professional assassin end sui...
45357,False,0.0,,227506,"In a small town live two brothers, one a minis...",0.003503,Yermoliev,Russia,1917-10-21,0.0,87.0,,,Satan Triumphant,0.0,0.0,,small town live two brother one minister one h...


In [27]:
indices = pd.Series(df.index, index = df['title']).drop_duplicates() # creating series for indices with movies

In [28]:
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Robin Hood                     45354
Century of Birthing            45355
Betrayal                       45356
Satan Triumphant               45357
Queerama                       45358
Length: 45359, dtype: int64

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [30]:
tfidf = TfidfVectorizer(ngram_range= (1,2), max_features = 50000, stop_words='english')

In [31]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [32]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1848000 stored elements and shape (45359, 50000)>

# **Cosine Similarities**

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
def recommend(title, n=10):
  if title not in indices:
    return ['Movie not found']

  idx = indices[title]
  sim_score = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
  similar_idx = sim_score.argsort()[::-1][1:n+1]
  return df['title'].iloc[similar_idx]

In [35]:
recommend('Avatar')

26519                                  Avatar 2
13557                      Dragonball Evolution
11913                     Live Free or Die Hard
10965                     X-Men: The Last Stand
11856    Fantastic 4: Rise of the Silver Surfer
13120                                 Australia
17572            Rise of the Planet of the Apes
11437                                    Eragon
44052     Avatar: Creating the World of Pandora
13626                  X-Men Origins: Wolverine
Name: title, dtype: str

In [36]:
import pickle

pickle.dump(tfidf_matrix,open('models/tfidf_matrix.pkl', 'wb'))

pickle.dump(indices,open('models/indices.pkl', 'wb'))

df.to_pickle('models/df.pkl')

pickle.dump(tfidf, open('models/tfidf.pkl', 'wb'))